## Registrando modelo

In [0]:
#Variaveis de Ambiente
try:
    SOURCE_CATALOG = dbutils.widgets.get("source_catalog")
except:
    SOURCE_CATALOG = "dev"

try:
    TARGET_CATALOG = dbutils.widgets.get("target_catalog")
except:
    TARGET_CATALOG = "test"

try:
    SCHEMA = dbutils.widgets.get("schema")
except:
    SCHEMA = "falha"

try:
    MODEL = dbutils.widgets.get("model")
except:
    MODEL = "failure_predictor"



In [0]:
import mlflow
from mlflow.tracking import MlflowClient
import mlflow.sklearn

In [0]:
EXPERIMENT_NAME = "/Shared/DataPulse/FailurePrediction"

In [0]:
import pandas as pd
def selecionar_melhor_run(
    experiment_name: str,
    metricas: list = None,
    ascending: bool = False
) -> pd.DataFrame:

    """
    Consulta todas as Runs de um experimento MLflow
    e classifica os modelos pelas métricas selecionadas.

    Critério padrão: média das métricas.
    """

    if metricas is None:
        metricas = [
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc"
        ]

    client = MlflowClient()

    experiment = client.get_experiment_by_name(
        experiment_name
    )

    if experiment is None:
        raise ValueError(
            f"Experimento não encontrado: {experiment_name}"
        )

    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string="attributes.status = 'FINISHED'",
        order_by=["attributes.start_time DESC"],
        max_results=10000
    )

    resultados = []

    for run in runs:

        metrics = run.data.metrics

        # Verifica se todas as métricas existem
        if not all(
            metric in metrics
            for metric in metricas
        ):
            continue

        resultado = {
            "run_id": run.info.run_id,
            "run_name": run.data.tags.get(
                "mlflow.runName", "N/A"
            ),
            "model": run.data.params.get(
                "model", "N/A"
            )
        }

        for metric in metricas:
            resultado[metric] = metrics[metric]

        # Média das métricas
        resultado["score"] = sum(
            metrics[metric]
            for metric in metricas
        ) / len(metricas)

        resultados.append(resultado)

    if not resultados:
        raise ValueError(
            "Nenhuma Run possui todas as métricas solicitadas."
        )

    df_resultados = pd.DataFrame(resultados)

    df_resultados = df_resultados.sort_values(
        by="score",
        ascending=ascending
    ).reset_index(drop=True)

    df_resultados.index += 1
    df_resultados.index.name = "ranking"

    return df_resultados

In [0]:
def validar_modelo(df_ranking):

    # Buscar os detalhes da execução (run), que incluem as métricas

    run_id = str(df_ranking["run_id"].iloc[0])
    model_uri = f"runs:/{run_id}/model"
    # Inicializar o cliente do MLflow
    client = MlflowClient()
    run = client.get_run(run_id)
    metrics = run.data.metrics
    print("Métricas:", metrics)
    # Validar o modelo


    if metrics["accuracy"] < 0.90:
        raise ValueError(
            "Modelo reprovado: Accuracy abaixo do mínimo"
        )

    if metrics["precision"] < 0.90:
        raise ValueError(
            "Modelo reprovado: precision abaixo do mínimo"
        )

    print("Modelo aprovado!")
    print("Registrando modelo!")
    #LogisticRegression_v2
    model_uri = f"runs:/{run_id}/model"

    source_name = f"{SOURCE_CATALOG}.{SCHEMA}.{MODEL}"
    target_name = f"{TARGET_CATALOG}.{SCHEMA}.{MODEL}"

    registered_model = mlflow.register_model(
        model_uri=model_uri,
        name=F"{source_name}",
    )    

    source_version = registered_model.version

    # Copiar artefato para o catálogo de destino
    new_version = client.copy_model_version(
        src_model_uri=(
            f"models:/{source_name}/{source_version}"
        ),
        dst_name=target_name
    )

    # Definir alias no ambiente de destino
    client.set_registered_model_alias(
        name=target_name,
        alias="Candidate",
        version=new_version.version
    )

    print(f"Modelo promovido: {target_name}")
    print(f"Versão: {new_version.version}")
    return True

In [0]:
df_ranking = selecionar_melhor_run(
    experiment_name=EXPERIMENT_NAME
)
display(df_ranking)
validar_modelo(df_ranking)

run_id,run_name,model,accuracy,precision,recall,f1,roc_auc,score
58a682b3c26442b187fe3d7f4601f0f8,GradientBoosting,N/A,0.9495798319327731,1.0,0.07692307692307693,0.14285714285714285,0.7008547008547008,0.5740429505135387
026cdfc161b34dc8a3f00f72313100e9,GaussianNB_v1,N/A,0.8991596638655462,0.21052631578947367,0.3076923076923077,0.25,0.6235897435897436,0.4581936061874142
c34353ef1a6d4c36a89703206fd55a2b,GaussianNB_v1,N/A,0.8991596638655462,0.21052631578947367,0.3076923076923077,0.25,0.6235897435897436,0.4581936061874142
b7b35d6e6397435bb704dda4ceff0c86,GaussianNB,N/A,0.8991596638655462,0.21052631578947367,0.3076923076923077,0.25,0.6235897435897436,0.4581936061874142
7471cd722f8d44c2ba786ed59a510e3f,GaussianNB,N/A,0.9074074074074074,0.18333333333333332,0.2018348623853211,0.19213973799126638,0.7575970976343,0.4484624877503257
82e2cb638eb343579e716b377394d160,DecisionTree,N/A,0.943943943943944,0.2,0.009174311926605505,0.017543859649122806,0.7024030966338192,0.37461304243069826
13e2d65fc01c4f7a875f338f56f76abb,GradientBoosting,N/A,0.9434434434434434,0.16666666666666666,0.009174311926605505,0.017391304347826087,0.7157323179586307,0.37048160886863446
0bc98c5e340845d1ad5ea041814eb75f,DecisionTree,N/A,0.9453781512605042,0.0,0.0,0.0,0.7822222222222223,0.3455200746965453
18e446f7afe24574902ef463026bc916,LogisticRegression,N/A,0.9454454454454454,0.0,0.0,0.0,0.7680244389293884,0.3426939768749667
f164b73b926246838ee04295133db66d,LogisticRegression_V2,N/A,0.9453781512605042,0.0,0.0,0.0,0.7442735042735042,0.3379303311068017


Métricas: {'accuracy': 0.9495798319327731, 'best_cv_score': 0.9218588503156887, 'f1': 0.14285714285714285, 'precision': 1.0, 'recall': 0.07692307692307693, 'roc_auc': 0.7008547008547008}
Modelo aprovado!
Registrando modelo!


Registered model 'dev.falha.failure_predictor' already exists. Creating a new version of this model...
2026/11/09 22:41:13 WARNING mlflow.tracking._model_registry.fluent: Run with id 58a682b3c26442b187fe3d7f4601f0f8 has no artifacts at artifact path 'model', registering model based on models:/m-ccafa4b29c9e4a02b04b3eb56172f3ef instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '2' of model 'dev.falha.failure_predictor': https://dbc-0395191b-4103.cloud.databricks.com/explore/data/models/dev/falha/failure_predictor/version/2?o=7474646727692125
Successfully registered model 'test.falha.failure_predictor'.


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

Copied version '2' of model 'dev.falha.failure_predictor' to version '1' of model 'test.falha.failure_predictor'.


Modelo promovido: test.falha.failure_predictor
Versão: 1


True